In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/CycPeptMPDB_Peptide_Assay_RRCK (1).csv')
df.shape

(186, 247)

In [3]:
last_id = df['ID'].max()
print('last_id: ', last_id)

last_id:  5668


In [4]:
#Step 1: Removing unreliable datapoints (datapoints with Permeability value <= -10) and selecting the required Columns
df = df[df['RRCK'] > -10]
selected_columns = ['ID', 'SMILES', 'RRCK', 'Sequence', 'MolWt']
new_df = df[selected_columns]
new_df.shape

(185, 5)

In [5]:
new_df.columns = ['ID', 'SMILES', 'Permeability', 'Sequence', 'MolWt']

In [6]:
new_df.head(10)

,ID,SMILES,Permeability,Sequence,MolWt
0,22,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.96,"['Abu', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', '...",1202.635
1,23,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[C...,-5.74,"['L', 'dL', 'L', 'L', 'dP', 'Y']",712.933
2,24,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](C...,-6.30,"['dL', 'L', 'L', 'L', 'dP', 'Y']",712.933
3,25,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...,-5.31,"['L', 'Me_dL', 'meL', 'L', 'dP', 'meY']",755.014
4,26,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H...,-5.39,"['Me_dL', 'meL', 'L', 'meL', 'dP', 'meY']",769.041
5,27,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N(C)[C@...,-5.46,"['meL', 'Me_dL', 'meL', 'meL', 'dP', 'meY']",783.068
6,28,CC(C)C[C@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)N2CCC[...,-5.21,"['Me_dL', 'meL', 'meL', 'meL', 'dP', 'meY']",783.068
7,29,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H...,-5.72,"['S', 'Me_dL', 'meL', 'L', 'dP', 'meY']",728.932
8,30,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...,-5.80,"['L', 'Me_dL', 'meS', 'L', 'dP', 'meY']",728.932
9,31,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...,-6.12,"['L', 'Me_dL', 'meL', 'S', 'dP', 'meY']",728.932


In [7]:
#Step 2: curating duplicate dapaoints based on SMILES representation
duplicates = new_df[new_df.duplicated(subset='SMILES', keep=False)]
non_duplicates = new_df[~new_df.duplicated(subset='SMILES', keep=False)]
print("DataFrame with duplicates:")
print(duplicates)
print("\nDataFrame without duplicates:")
print(non_duplicates)
duplicates.to_csv('/home/users/akshay/PCPpred/RRCK/data/duplicates_RRCK.csv', index=False)

DataFrame with duplicates:
       ID                                             SMILES  Permeability  \
0      22  C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...         -5.96   
1      23  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[C...         -5.74   
3      25  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...         -5.31   
8      30  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...         -5.80   
10     32  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...         -5.31   
11     33  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...         -5.81   
14     36  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...         -5.26   
21     43  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...         -6.74   
39    980  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...         -5.31   
40    981  C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...         -5.96   
68   1862  C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...         -5.25   
122  2326  CC(C)C[C@@H]1NC(=O)[C@H](C

In [8]:
duplicates = new_df[new_df.duplicated(subset='SMILES', keep=False)]

non_duplicates = new_df[~new_df.duplicated(subset='SMILES', keep=False)]
#Permeability and MolWt of duplicate SMILES were averaged, and Most abundandant sequence was chosen
grouped_duplicates = duplicates.groupby('SMILES').agg(
    Permeability=('Permeability', 'mean'),   
    MolWt=('MolWt', 'mean'),   
    Sequence=('Sequence', lambda x: x.mode()[0]), 
).reset_index()

#Unique ID's were given to these curated datapoints, where ID's start from the last documented id in CycPeptMPDB_Peptide_Assay_PAMPA (5)
grouped_duplicates['ID'] = range(last_id + 1, last_id + 1 + len(grouped_duplicates))

grouped_duplicates = grouped_duplicates[['ID', 'SMILES', 'Permeability', 'Sequence', 'MolWt']]

non_duplicates_ordered = non_duplicates[['ID', 'SMILES', 'Permeability', 'Sequence', 'MolWt']]
merged_df = pd.concat([non_duplicates_ordered, grouped_duplicates], ignore_index=True)

print("DataFrame with processed duplicates:")
print(grouped_duplicates)

print("\nDataFrame without duplicates:")
print(non_duplicates)

print("Merged DataFrame:")
print(merged_df)

grouped_duplicates.to_csv('/home/users/akshay/PCPpred/RRCK/data/grouped_duplicates_RRCK.csv', index=False)
non_duplicates.to_csv('/home/users/akshay/PCPpred/RRCK/data/non_duplicates_RRCK.csv', index=False)
merged_df.to_csv('/home/users/akshay/PCPpred/RRCK/data/CycPeptMPDB_data_Main_RRCK.csv', index=False)

DataFrame with processed duplicates:
     ID                                             SMILES  Permeability  \
0  5669  C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...       -5.7600   
1  5670  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...       -5.2250   
2  5671  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...       -5.2025   
3  5672  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...       -6.1850   
4  5673  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...       -5.8050   
5  5674  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[C...       -5.8050   

                                            Sequence     MolWt  
0  ['Abu', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', '...  1202.635  
1            ['L', 'Me_dL', 'meA', 'L', 'dP', 'meY']   712.933  
2            ['L', 'Me_dL', 'meL', 'L', 'dP', 'meY']   755.014  
3         ['L', 'Me_dL', 'Me_Cha', 'L', 'dP', 'meY']   795.079  
4            ['L', 'Me_dL', 'meS', 'L', 'dP', 'meY']   728.932  
5                   ['L', 'dL', 'L', 'L'

In [9]:
#Step 3: Splitting the Main dataset into Train and Test in ration 0.8:0.2
df = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/CycPeptMPDB_data_Main_RRCK.csv')
df

,ID,SMILES,Permeability,Sequence,MolWt
0,24,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](C...,-6.3000,"['dL', 'L', 'L', 'L', 'dP', 'Y']",712.933
1,26,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H...,-5.3900,"['Me_dL', 'meL', 'L', 'meL', 'dP', 'meY']",769.041
2,27,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N(C)[C@...,-5.4600,"['meL', 'Me_dL', 'meL', 'meL', 'dP', 'meY']",783.068
3,28,CC(C)C[C@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)N2CCC[...,-5.2100,"['Me_dL', 'meL', 'meL', 'meL', 'dP', 'meY']",783.068
4,29,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H...,-5.7200,"['S', 'Me_dL', 'meL', 'L', 'dP', 'meY']",728.932
...,...,...,...,...,...
170,5670,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...,-5.2250,"['L', 'Me_dL', 'meA', 'L', 'dP', 'meY']",712.933
171,5671,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...,-5.2025,"['L', 'Me_dL', 'meL', 'L', 'dP', 'meY']",755.014
172,5672,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...,-6.1850,"['L', 'Me_dL', 'Me_Cha', 'L', 'dP', 'meY']",795.079
173,5673,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...,-5.8050,"['L', 'Me_dL', 'meS', 'L', 'dP', 'meY']",728.932


In [10]:
#Main dataset is sorted based on Molecular weight in descending order
df_main = df.sort_values(by='MolWt', ascending= False)
df_main

,ID,SMILES,Permeability,Sequence,MolWt
132,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.34,"['T', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1218.634
137,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,"['V', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1216.662
138,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,"['V', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1214.646
169,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,"['Abu', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', '...",1202.635
139,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,"['Me_Bmt(E)', 'Abu', 'Sar', 'meL', 'V', 'meL',...",1202.635
...,...,...,...,...,...
90,2306,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H]2CCCN...,-4.75,"['L', 'Me_dL', 'meA', 'L', 'dP', 'meA']",620.836
115,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,"['L', 'dA', 'A', 'L', 'dP', 'F']",612.772
89,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,"['L', 'Me_dNva', 'meA', 'L', 'dP', 'meA']",606.809
88,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,"['L', 'Me_dAbu', 'meA', 'L', 'dP', 'meA']",592.782


In [11]:
df_main['MolWt']

132    1218.634
137    1216.662
138    1214.646
169    1202.635
139    1202.635
         ...   
90      620.836
115     612.772
89      606.809
88      592.782
87      578.755
Name: MolWt, Length: 175, dtype: float64

In [12]:
#Every fifth datapoint is allocated to test set and remaining datapoints to Train set
test_mask = np.arange(len(df_main)) % 5 == 0
test_df = df_main[test_mask]
test_df.shape

(35, 5)

In [13]:
train_df = df_main[~test_mask]
train_df.shape

(140, 5)

In [14]:
print(train_df["Permeability"].max())
print(train_df["Permeability"].min())
print(train_df.shape)

-4.51
-7.0
(140, 5)


In [15]:
print(test_df["Permeability"].max())
print(test_df["Permeability"].min())
print(test_df.shape)

-4.49
-7.0
(35, 5)


In [16]:
test_df.to_csv("/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.csv",index=False)
train_df.to_csv("/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.csv",index=False)

In [17]:
train_df

,ID,SMILES,Permeability,Sequence,MolWt
137,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,"['V', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1216.662
138,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,"['V', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1214.646
169,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,"['Abu', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', '...",1202.635
139,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,"['Me_Bmt(E)', 'Abu', 'Sar', 'meL', 'V', 'meL',...",1202.635
133,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,"['A', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1188.608
...,...,...,...,...,...
116,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,"['L', 'dAbu', 'A', 'L', 'dP', 'F']",626.799
115,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,"['L', 'dA', 'A', 'L', 'dP', 'F']",612.772
89,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,"['L', 'Me_dNva', 'meA', 'L', 'dP', 'meA']",606.809
88,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,"['L', 'Me_dAbu', 'meA', 'L', 'dP', 'meA']",592.782


In [18]:
test_df

,ID,SMILES,Permeability,Sequence,MolWt
132,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,"['T', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1218.634
136,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,"['Abu', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', '...",1202.635
78,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,"['Me_dA', 'Me_Nva', 'Me_dAbu', 'Me_Nle', 'Me_d...",1095.438
77,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,"['Me_dA', 'Me_Nva', 'Me_dAbu', 'meA', 'Me_dAbu...",1039.330
74,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,"['Me_Nva', 'Me_dAbu', 'meA', 'Me_dNle', 'Me_dA...",996.305
63,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,"['Me_dA', 'meA', 'Me_dL', 'Me_dL', 'meL', 'Me_...",953.280
71,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,"['Me_dAbu', 'Me_Nva', 'Me_dNva', 'Me_dNva', 'M...",939.253
55,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,"['Ala(tBu)', 'Tza', 'dP', 'Mono48', 'Ser(Bn)',...",914.089
162,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,"['F', 'Me_Nle', 'P', 'Me_Nle', 'Nle', 'Nle', '...",899.213
49,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,"['Ala(tBu)', 'Tza', 'dP', 'PhPr_Gly', 'Nva(Ph)...",876.137


In [19]:
import pandas as pd

df = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.csv')

with open('/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.smi', 'w') as f:
    for _, row in df.iterrows():
        smiles = row['SMILES']
        id = row['ID']
        f.write(f"{smiles} {id}\n")
print("train.smi file has been created successfully.")

train.smi file has been created successfully.


In [20]:
df = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.csv')

with open('/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.smi', 'w') as f:
    for _, row in df.iterrows():
        smiles = row['SMILES']
        id = row['ID']
        f.write(f"{smiles} {id}\n")
print("test.smi file has been created successfully.")

test.smi file has been created successfully.
